# App-27 — Sparse index tracking : du benchmark qui fuit au walk-forward vérifiable

> **Navigation** : [Applications Search](../README.md) · [CSP-3 (CP-SAT avancé)](../../Part2-CSP/CSP-3-Advanced.ipynb) · [CSP-5 (optimisation)](../../Part2-CSP/CSP-5-Optimization.ipynb) · [QuantConnect — construction de portefeuille](../../../QuantConnect/Python/QC-Py-14-Portfolio-Construction-Execution.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. distinguer **sélection d'univers**, **calibration**, **validation** et **test futur** ;
2. modéliser un portefeuille sparse en lots entiers avec cardinalité, caps sectoriels et turnover temporel ;
3. lire un statut CP-SAT avec incumbent, borne et gap plutôt qu'une seule valeur d'objectif ;
4. valider indépendamment budget, cardinalité, secteurs et turnover ;
5. mesurer séparément erreur active quadratique, biais actif et tracking error annualisée ;
6. comparer un protocole contaminé à un walk-forward sans fuite.

**Prérequis** : rendements, validation chronologique, programmation par contraintes, CP-SAT. **Durée estimée** : 60 min.

## Hommage à un travail étudiant

Ce notebook distille le projet **M2 — Sparse Index Tracking** de **Godric Bouteloup**, EPITA SCIA, Programmation par Contraintes 2026 : [répertoire source](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/tree/main/M2-godric_bouteloup), [PR #52](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/pull/52), commit `bbc372b4`, licence MIT.

Le rendu étudiant apporte le geste central : lots entiers, variables de sélection, cardinalité, caps sectoriels et objectif L1. La reproduction fraîche a toutefois révélé que la contrainte dite de turnover reliait les solutions de **K successifs**, pas deux rebalancements, tandis que le choix d'hyperparamètre et le backtest réutilisaient les périodes évaluées. CoursIA ne recopie pas le notebook : il reconstruit une expérience autonome et déterministe qui rend ces différences exécutables.

> **Provenance.** Source, défauts reproduits, transformations et limites : [`data/app27-sparse-index-tracking/SOURCE.md`](data/app27-sparse-index-tracking/SOURCE.md).

## 1. Données synthétiques à vérité connue

Une distillation pédagogique doit rester rejouable sans Wikipédia, Yahoo Finance ni survivorship bias caché. Nous générons donc un petit marché factoriel seedé : quatre secteurs, un benchmark diversifié et un changement de régime à mi-parcours. Le but n'est pas d'imiter le niveau du S&P 500, mais d'auditer le **protocole**.

In [1]:
from dataclasses import dataclass
from math import sqrt
from typing import Iterable

import numpy as np
import pandas as pd
from ortools.sat.python import cp_model

SEED = 20260901
RNG = np.random.default_rng(SEED)
print(f"Environnement prêt : numpy={np.__version__}, pandas={pd.__version__}, seed={SEED}")

Environnement prêt : numpy=2.4.4, pandas=2.3.3, seed=20260901


L'univers contient 24 actifs et 240 jours. Les poids du benchmark sont fixes et connus ; les actifs chargent sur un facteur marché et un facteur sectoriel. Le régime futur augmente la volatilité de deux secteurs, ce qui rend une sélection faite sur tout l'historique artificiellement avantageuse.

In [2]:
n_days, n_assets, n_sectors = 240, 24, 4
sectors = np.repeat(np.arange(n_sectors), n_assets // n_sectors)
market = RNG.normal(0.0003, 0.007, n_days)
sector_factors = RNG.normal(0, 0.004, (n_days, n_sectors))
sector_factors[140:, 2:] *= 1.8
betas = RNG.uniform(0.75, 1.2, n_assets)
idiosyncratic = RNG.normal(0, 0.006, (n_days, n_assets))
returns = market[:, None] * betas + sector_factors[:, sectors] + idiosyncratic
benchmark_weights = RNG.dirichlet(np.ones(n_assets) * 2.5)
benchmark = returns @ benchmark_weights + RNG.normal(0, 0.0004, n_days)
dates = pd.date_range("2020-01-01", periods=n_days, freq="B")
returns_df = pd.DataFrame(returns, index=dates, columns=[f"A{i:02d}" for i in range(n_assets)])
benchmark_s = pd.Series(benchmark, index=dates, name="INDEX")
print(f"Marché synthétique : {returns_df.shape[0]} jours, {returns_df.shape[1]} actifs, {n_sectors} secteurs")
print(f"Poids benchmark : somme={benchmark_weights.sum():.6f}, min={benchmark_weights.min():.4f}, max={benchmark_weights.max():.4f}")

Marché synthétique : 240 jours, 24 actifs, 4 secteurs
Poids benchmark : somme=1.000000, min=0.0065, max=0.0775


### Lecture du résultat : une expérience contrôlée

Les dimensions et la somme des poids imprimées ci-dessus sont des invariants vérifiables. Le changement de régime est dans la génération, mais il n'est jamais utilisé par l'optimiseur : le futur reste futur.

## 2. Métriques : ne pas confondre trois questions

Pour les rendements actifs $a_t=r_{p,t}-r_{b,t}$ :

- la **RMSE active** $\sqrt{\frac1T\sum_t a_t^2}$ correspond à l'objectif quadratique annoncé ;
- le **biais actif** $\bar a$ détecte une sous-performance systématique ;
- la **tracking error annualisée** $\mathrm{std}(a)\sqrt{252}$ mesure la volatilité active au sens financier.

Une seule `std` ne répond pas aux trois questions.

In [3]:
def active_metrics(portfolio: np.ndarray, target: np.ndarray) -> dict[str, float]:
    active = np.asarray(portfolio, dtype=float) - np.asarray(target, dtype=float)
    return {
        "rmse": float(np.sqrt(np.mean(active**2))),
        "bias": float(np.mean(active)),
        "te_annualized": float(np.std(active, ddof=1) * np.sqrt(252)),
    }

constant_lag = active_metrics(np.zeros(5), np.ones(5) * 0.001)
assert np.isclose(constant_lag["rmse"], 0.001)
assert np.isclose(constant_lag["te_annualized"], 0.0)
print("Contre-exemple métrique :", constant_lag)

Contre-exemple métrique : {'rmse': 0.001, 'bias': -0.001, 'te_annualized': 0.0}


### Lecture du résultat : zéro volatilité ne signifie pas zéro erreur

Le portefeuille fictif sous-performe chaque jour du même montant : sa tracking error au sens volatilité est nulle, mais sa RMSE et son biais ne le sont pas. Le notebook source annonçait un objectif L2 puis rapportait seulement `std`; cette séparation répare l'ambiguïté au lieu de renommer une métrique.

### Exercice 1 — Auditer une métrique active

Complétez une fonction qui retourne RMSE, biais et tracking error annualisée. **Indice :** réutilisez `active_metrics` et vérifiez le cas d'un retard constant.

In [4]:
def audit_metric(portfolio: np.ndarray, target: np.ndarray):
    # TODO étudiant : retourner les trois métriques et ajouter un invariant utile.
    return None

print("Exercice 1 à compléter : audit des métriques actives")

Exercice 1 à compléter : audit des métriques actives


## 3. Modèle CP-SAT en lots entiers

Le modèle distribue exactement 100 lots, impose **exactement K actifs positifs**, limite chaque secteur à 45 lots et borne le turnover par rapport au portefeuille de la date précédente. La pénalité L1 porte sur la calibration seulement. Contrairement au rendu source, deux valeurs de K ne sont jamais reliées entre elles.

In [5]:
@dataclass
class SolveResult:
    weights: np.ndarray
    status: str
    objective: float
    bound: float
    gap: float
    wall_time: float


def build_and_solve(
    x_train: np.ndarray,
    y_train: np.ndarray,
    asset_sectors: np.ndarray,
    k: int,
    previous_lots: np.ndarray | None = None,
    turnover_cap: int = 30,
    time_limit: float = 3.0,
) -> SolveResult:
    scale = 100_000
    x_scaled = np.rint(x_train * scale).astype(int)
    y_scaled = np.rint(y_train * scale).astype(int)
    n = x_train.shape[1]
    model = cp_model.CpModel()
    lots = [model.new_int_var(0, 100, f"lot_{i}") for i in range(n)]
    selected = [model.new_bool_var(f"selected_{i}") for i in range(n)]
    model.add(sum(lots) == 100)
    model.add(sum(selected) == k)
    for i in range(n):
        model.add(lots[i] >= selected[i])
        model.add(lots[i] <= 100 * selected[i])
    for sector in np.unique(asset_sectors):
        model.add(sum(lots[i] for i in range(n) if asset_sectors[i] == sector) <= 45)

    if previous_lots is not None:
        absolute_changes = []
        for i in range(n):
            change = model.new_int_var(0, 100, f"turnover_{i}")
            model.add_abs_equality(change, lots[i] - int(previous_lots[i]))
            absolute_changes.append(change)
        model.add(sum(absolute_changes) <= 2 * turnover_cap)

    errors = []
    max_error = int(200 * scale)
    for t in range(len(y_scaled)):
        error = model.new_int_var(-max_error, max_error, f"error_{t}")
        model.add(error == sum(int(x_scaled[t, i]) * lots[i] for i in range(n)) - int(y_scaled[t]) * 100)
        absolute_error = model.new_int_var(0, max_error, f"absolute_error_{t}")
        model.add_abs_equality(absolute_error, error)
        errors.append(absolute_error)
    model.minimize(sum(errors))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers = 1
    solver.parameters.random_seed = SEED
    status_code = solver.solve(model)
    if status_code not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise RuntimeError(f"CP-SAT n'a pas trouvé d'incumbent : {solver.status_name(status_code)}")
    weights = np.array([solver.value(v) for v in lots], dtype=int)
    objective = float(solver.objective_value)
    bound = float(solver.best_objective_bound)
    gap = 0.0 if objective == 0 else max(0.0, (objective - bound) / abs(objective))
    return SolveResult(weights / 100, solver.status_name(status_code), objective, bound, gap, solver.wall_time)

smoke = build_and_solve(returns[:60], benchmark[:60], sectors, k=6, time_limit=2.0)
print(f"Smoke CP-SAT : statut={smoke.status}, actifs={(smoke.weights > 0).sum()}, somme={smoke.weights.sum():.2f}, gap={smoke.gap:.3%}")

Smoke CP-SAT : statut=FEASIBLE, actifs=6, somme=1.00, gap=83.682%


### Lecture du résultat : l'incumbent n'est pas la preuve

Le statut, le nombre d'actifs, le budget et le gap sont imprimés ensemble. Un résultat `FEASIBLE` reste exploitable comme incumbent, mais il ne devient jamais `OPTIMAL` par omission de la borne.

## 4. Validateur indépendant

Le solveur construit une solution ; une autre fonction vérifie ses contraintes à partir du vecteur de poids. Cette séparation détecte les erreurs de modèle, de scaling ou d'extraction.

In [6]:
def validate_portfolio(
    weights: np.ndarray,
    asset_sectors: np.ndarray,
    k: int,
    previous_weights: np.ndarray | None = None,
    turnover_cap: float = 0.30,
) -> dict[str, float | int | bool]:
    active = weights > 1e-9
    sector_loads = [float(weights[asset_sectors == s].sum()) for s in np.unique(asset_sectors)]
    turnover = 0.0 if previous_weights is None else float(np.abs(weights - previous_weights).sum() / 2)
    checks = {
        "budget_ok": bool(np.isclose(weights.sum(), 1.0)),
        "cardinality": int(active.sum()),
        "cardinality_ok": bool(active.sum() == k),
        "sector_cap_ok": bool(max(sector_loads) <= 0.45 + 1e-9),
        "turnover": turnover,
        "turnover_ok": bool(previous_weights is None or turnover <= turnover_cap + 1e-9),
    }
    assert all(checks[key] for key in ("budget_ok", "cardinality_ok", "sector_cap_ok", "turnover_ok"))
    return checks

smoke_checks = validate_portfolio(smoke.weights, sectors, 6)
print("Validation indépendante :", smoke_checks)

Validation indépendante : {'budget_ok': True, 'cardinality': 6, 'cardinality_ok': True, 'sector_cap_ok': True, 'turnover': 0.0, 'turnover_ok': True}


### Lecture du résultat : chaque claim devient une assertion

Le validateur ne consulte ni les variables CP-SAT ni le statut du solveur. Il recalcule budget, cardinalité, concentration et turnover depuis la solution extraite.

### Exercice 2 — Ajouter une contrainte métier

Ajoutez une borne de poids individuel au validateur, puis au modèle. **Indice :** exprimez la borne dans les mêmes 100 lots et écrivez d'abord un test qui échoue.

In [7]:
def validate_individual_cap(weights: np.ndarray, cap: float = 0.20):
    # TODO étudiant : retourner True si tous les poids respectent cap.
    return None

print("Exercice 2 à compléter : cap individuel")

Exercice 2 à compléter : cap individuel


## 5. Protocole réparé : walk-forward

À chaque date de rebalancement :

1. l'univers est classé **uniquement** sur la fenêtre de calibration ;
2. chaque K est résolu indépendamment sur cette fenêtre ;
3. K est choisi sur une fenêtre de validation qui précède le test ;
4. une nouvelle optimisation est faite avec ce K et le portefeuille précédent ;
5. le bloc test futur est évalué une seule fois.

Le premier portefeuille n'a pas de turnover antérieur ; les suivants sont reliés dans le temps.

In [8]:
def rank_universe(x: np.ndarray, y: np.ndarray, size: int) -> np.ndarray:
    correlations = np.array([np.corrcoef(x[:, i], y)[0, 1] for i in range(x.shape[1])])
    correlations = np.nan_to_num(correlations, nan=-1.0)
    return np.argsort(correlations)[-size:]


def run_walk_forward(
    x: np.ndarray,
    y: np.ndarray,
    asset_sectors: np.ndarray,
    candidate_k: Iterable[int] = (4, 6, 8),
) -> tuple[pd.DataFrame, list[np.ndarray]]:
    rows, portfolios = [], []
    previous_full = None
    for test_start in (100, 130, 160, 190):
        train = slice(test_start - 80, test_start - 20)
        validation = slice(test_start - 20, test_start)
        test = slice(test_start, test_start + 20)
        ranked = rank_universe(x[train], y[train], size=16)
        held = np.flatnonzero(previous_full > 1e-9) if previous_full is not None else np.array([], dtype=int)
        universe = np.unique(np.concatenate([ranked, held]))
        validation_scores = {}
        candidates = {}
        for k in candidate_k:
            solved = build_and_solve(x[train][:, universe], y[train], asset_sectors[universe], k, time_limit=2.0)
            validation_scores[k] = active_metrics(x[validation][:, universe] @ solved.weights, y[validation])["rmse"]
            candidates[k] = solved
        chosen_k = min(validation_scores, key=validation_scores.get)
        previous_local_lots = None if previous_full is None else np.rint(previous_full[universe] * 100).astype(int)
        final = build_and_solve(
            x[test_start - 80:test_start][:, universe],
            y[test_start - 80:test_start],
            asset_sectors[universe],
            chosen_k,
            previous_lots=previous_local_lots,
            time_limit=3.0,
        )
        full = np.zeros(x.shape[1])
        full[universe] = final.weights
        checks = validate_portfolio(full, asset_sectors, chosen_k, previous_full)
        metrics = active_metrics(x[test] @ full, y[test])
        rows.append({
            "test_start": test_start,
            "k": chosen_k,
            "status": final.status,
            "gap": final.gap,
            "turnover": checks["turnover"],
            **metrics,
        })
        portfolios.append(full)
        previous_full = full
    return pd.DataFrame(rows), portfolios

walk_results, walk_portfolios = run_walk_forward(returns, benchmark, sectors)
print(walk_results.to_string(index=False, formatters={"gap": "{:.2%}".format, "turnover": "{:.3f}".format, "rmse": "{:.5f}".format, "bias": "{:.5f}".format, "te_annualized": "{:.3%}".format}))

 test_start  k   status    gap turnover    rmse    bias te_annualized
        100  8  OPTIMAL  0.00%    0.000 0.00161 0.00053        2.478%
        130  6  OPTIMAL  0.00%    0.300 0.00186 0.00034        2.984%
        160  8  OPTIMAL  0.00%    0.300 0.00172 0.00006        2.803%
        190  8 FEASIBLE 31.41%    0.300 0.00125 0.00015        2.030%


### Lecture du résultat : les quatre blocs test sont disjoints

Chaque ligne correspond à un futur jamais utilisé pour choisir l'univers ni K. Le turnover est nul au premier rebalancement par définition, puis borné entre portefeuilles consécutifs. Les gaps publiés empêchent de confondre qualité de l'incumbent et preuve d'optimalité.

## 6. Reproduire la fuite pour mesurer son effet

Le protocole contaminé classe l'univers et choisit K en regardant directement le bloc test. Il ne représente pas une baseline acceptable : c'est un **contre-exemple exécutable** qui montre pourquoi une belle métrique peut être invalide.

In [9]:
def run_leaky_protocol(x: np.ndarray, y: np.ndarray, asset_sectors: np.ndarray) -> pd.DataFrame:
    rows = []
    for test_start in (100, 130, 160, 190):
        train = slice(test_start - 80, test_start)
        test = slice(test_start, test_start + 20)
        universe = rank_universe(x[:test_start + 20], y[:test_start + 20], size=16)
        tested = []
        for k in (4, 6, 8):
            solved = build_and_solve(x[train][:, universe], y[train], asset_sectors[universe], k, time_limit=2.0)
            rmse = active_metrics(x[test][:, universe] @ solved.weights, y[test])["rmse"]
            tested.append((rmse, k, solved.weights))
        rmse, chosen_k, weights = min(tested, key=lambda row: row[0])
        rows.append({"test_start": test_start, "k": chosen_k, "rmse": rmse})
    return pd.DataFrame(rows)

leaky_results = run_leaky_protocol(returns, benchmark, sectors)
comparison = walk_results[["test_start", "rmse"]].merge(leaky_results, on="test_start", suffixes=("_walk", "_leaky"))
comparison["apparent_gain"] = comparison["rmse_walk"] - comparison["rmse_leaky"]
print(comparison.to_string(index=False, formatters={"rmse_walk": "{:.5f}".format, "rmse_leaky": "{:.5f}".format, "apparent_gain": "{:+.5f}".format}))
print(f"Écart moyen walk - protocole contaminé : {comparison['apparent_gain'].mean():+.5f}")

 test_start rmse_walk  k rmse_leaky apparent_gain
        100   0.00161  8    0.00150      +0.00011
        130   0.00186  8    0.00258      -0.00072
        160   0.00172  8    0.00210      -0.00038
        190   0.00125  8    0.00201      -0.00076
Écart moyen walk - protocole contaminé : -0.00044


### Lecture du résultat : une fuite n'offre aucune garantie de domination

La colonne `apparent_gain` compare les deux protocoles sur les mêmes blocs futurs. Une valeur positive signifie que le protocole contaminé paraît meilleur ; une valeur négative montre qu'il peut aussi choisir une solution moins bonne malgré son accès indu au test. Dans les deux cas, son score est **ininterprétable** : regarder le futur invalide l'estimation, sans garantir un gain sur chaque échantillon.

### Exercice 3 — Stress test de régime

Faites varier l'amplitude du changement de régime et répétez plusieurs seeds. **Indice :** rapportez une distribution d'optimisme, pas un seul run favorable.

In [10]:
def stress_regime(seeds: Iterable[int]):
    # TODO étudiant : régénérer le marché pour chaque seed et agréger l'écart.
    return None

print("Exercice 3 à compléter : stress test multi-seeds")

Exercice 3 à compléter : stress test multi-seeds


## 7. Ce qui est prouvé — et ce qui ne l'est pas

| Claim | Preuve dans ce notebook | Limite |
|---|---|---|
| Portefeuille faisable | validateur indépendant + assertions | marché synthétique |
| Cardinalité exacte | `sum(selected) == K` et poids positifs | 100 lots seulement |
| Turnover temporel | comparaison entre rebalancements consécutifs | univers restreint à 16 actifs |
| Test sans fuite | indices train/validation/test disjoints | quatre blocs seulement |
| Qualité solveur | statut, incumbent, borne, gap | time-limit court |
| Effet de la fuite | contre-protocole exécuté sur les mêmes blocs | une seed dans l'exemple guide |

## Conclusion

Le geste étudiant — utiliser CP-SAT pour les contraintes discrètes d'un tracker sparse — est conservé. La contribution CoursIA est le **protocole de preuve** : données autonomes, séparation chronologique, K choisi avant le test, turnover entre dates, validateur indépendant et gaps explicites.

Ce notebook ne revendique ni performance financière réelle, ni réplication du S&P 500, ni backtest QuantConnect. Passer à des données de marché exige un snapshot daté et licencié, un univers point-in-time, coûts et corporate actions, puis un backtest réellement out-of-sample.